#### Prepare the tables. We use shampoo dataset and split it into train and test sets.

In [1]:
# First we prepare the data for the scenario.
from hana_ml import dataframe
from hana_ml.algorithms.pal.utility import DataSets

connection_context = dataframe.ConnectionContext(userkey="RaysKey")

ts_df = DataSets.load_shampoo_data(connection_context)

Table SHAMPOO_SALES_DATA_TBL exists.


In [2]:
ts_df.head(30).save("SHAMPOO_SALES_TRAIN", force=True)
ts_df.tail(6).save("SHAMPOO_SALES_ACTUAL", force=True)
ts_df.tail(6).select(ts_df.columns[0]).save("SHAMPOO_SALES_PREDICT", force=True)

#### Setup Chatbot with HANA ML Toolkit

In [8]:
from gen_ai_hub.proxy.langchain import init_llm
from hana_ai.agents.hanaml_agent_with_memory import HANAMLAgentWithMemory
from hana_ai.tools.toolkit import HANAMLToolkit


llm = init_llm('gpt-4-32k', temperature=0.0, max_tokens=400)
tools = HANAMLToolkit(connection_context, used_tools='all', return_direct={"ts_dataset_report": False}).set_bas().get_tools()
chatbot = HANAMLAgentWithMemory(llm=llm, tools=tools, session_id='hana_ai_test', n_messages=10, verbose=True)

In [4]:
for tool in tools:
    print(tool.name)
    print(tool.return_direct)

accuracy_measure
False
additive_model_forecast_fit_and_save
False
additive_model_forecast_load_model_and_predict
False
automatic_timeseries_fit_and_save
False
automatic_timeseries_load_model_and_predict
False
automatic_timeseries_load_model_and_score
False
cap_artifacts
False
fetch_data
False
forecast_line_plot
False
intermittent_forecast
False
list_models
False
ts_dataset_report
False
ts_check
False
ts_outlier_detection
False


##### Fetch the data. The tool `fetch_data` in the toolkit is used to fetch the data from the HANA database. The data is returned in the JSON format.

In [9]:
chatbot.run("Show me the last 10 records of data from SHAMPOO_SALES_DATA_TBL")




> Entering new AgentExecutor chain...
Action:
{
  "action": "fetch_data",
  "action_input": {
    "table_name": "SHAMPOO_SALES_DATA_TBL",
    "last_n": 10
  }
}

> Finished chain.


,ID,SALES
0,27,315.9
1,28,439.3
2,29,401.3
3,30,437.4
4,31,575.5
5,32,407.6
6,33,682.0
7,34,475.3
8,35,581.3
9,36,646.9


##### Create a dataset report for the given data using the tool `ts_dataset_report`

In [10]:
chatbot.run("Then maybe create a dataset report for me on this table")



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "ts_dataset_report",
  "action_input": {
    "table_name": "SHAMPOO_SALES_DATA_TBL",
    "key": "ID",
    "endog": "SALES"
  }
}
```

100%|██████████| 8/8 [00:19<00:00,  2.45s/it]



Observation: {"html_file": "C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_report\\SHAMPOO_SALES_DATA_TBL_ts_report.html"}
Thought:The dataset report has been generated successfully. The report is saved as an HTML file. You can view the report by opening the HTML file in a web browser. The file is located at "C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_report\\SHAMPOO_SALES_DATA_TBL_ts_report.html".

> Finished chain.


'The dataset report has been generated successfully. The report is saved as an HTML file. You can view the report by opening the HTML file in a web browser. The file is located at "C:\\\\Users\\\\I308290\\\\AppData\\\\Local\\\\Temp\\\\hanaml_report\\\\SHAMPOO_SALES_DATA_TBL_ts_report.html".'

##### Use `ts_check` to calculate the statistics of the dataset.

In [9]:
print(chatbot.run("I want to check the time series data SHAMPOO_SALES_TRAIN, the key is ID and the target is SALES"))



> Entering new AgentExecutor chain...
Thought: The user is asking to check the time series data again. I will call the ts_check tool with the provided table name, key, and target.
Action:
```
{
  "action": "ts_check",
  "action_input": {
    "table_name": "SHAMPOO_SALES_TRAIN",
    "key": "ID",
    "endog": "SALES"
  }
}
```
Observation: Intermittent Test: proportion of zero values is 0.0
Stationarity Test: The stationary is 0.The kpss_stat is 0.428592.The p-value is 0.06483091698669324.The lags is 8.The number of observations is 30.The critical values is {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}.
Trend Test: Upward trend.
Seasonality Test: The type is additive.The period is 2.The acf is 0.553738.
Thought:The time series data from the table "SHAMPOO_SALES_TRAIN" has been checked. The results indicate that there are no zero values in the data. The KPSS test for stationarity gives a p-value of 0.0648, which is greater than 0.05, suggesting that the series is stationary. T

##### Model suggestion

In [10]:
chatbot.run("could you suggest the model for me based on the check result?")



> Entering new AgentExecutor chain...
Thought: Based on the check result, the data is stationary, has an upward trend, and shows additive seasonality with a period of 2. Therefore, an additive model forecast might be a good fit for this data. However, the final decision should be based on further model fitting and evaluation. 

Action:
```
{
  "action": "Final Answer",
  "action_input": "Based on the check result, an additive model forecast might be a good fit for this data. It can handle the upward trend and additive seasonality in your data. However, the final decision should be based on further model fitting and evaluation."
}
```

> Finished chain.


'Based on the check result, an additive model forecast might be a good fit for this data. It can handle the upward trend and additive seasonality in your data. However, the final decision should be based on further model fitting and evaluation.'

##### Use suggested model to train the model via the tool `xxx_fit_and_save`. 

In [11]:
chatbot.run("Then please train the table using the suggested model maybe automatic time series for me and save as my_automl_model, the key is ID and target is SALES")



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "automatic_timeseries_fit_and_save",
  "action_input": {
    "fit_table": "SHAMPOO_SALES_TRAIN",
    "name": "my_automl_model",
    "key": "ID",
    "endog": "SALES"
  }
}
```
Observation: {"trained_table": "SHAMPOO_SALES_TRAIN", "model_storage_name": "my_automl_model", "model_storage_version": 1}
Thought:The model has been successfully trained on the table 'SHAMPOO_SALES_TRAIN' and saved under the name 'my_automl_model' with version 1 in the model storage.

> Finished chain.


"The model has been successfully trained on the table 'SHAMPOO_SALES_TRAIN' and saved under the name 'my_automl_model' with version 1 in the model storage."

##### Predict on the model using `xxx_load_model_and_predict` tool.

In [12]:
chatbot.run("I want to predict the SHAMPOO_SALES_PREDICT using the trained model")



> Entering new AgentExecutor chain...
Thought: The user wants to predict the SHAMPOO_SALES_PREDICT using the trained model. I will use the automatic_timeseries_load_model_and_predict tool to do this. The required parameters for this tool are the name of the model and the table to predict. The name of the model is 'my_automl_model' and the table to predict is 'SHAMPOO_SALES_PREDICT'. The key of the dataset is 'ID'.
Action:
```
{
  "action": "automatic_timeseries_load_model_and_predict",
  "action_input": {
    "name": "my_automl_model",
    "predict_table": "SHAMPOO_SALES_PREDICT",
    "key": "ID"
  }
}
```
Observation: {"predicted_results_table": "my_automl_model_1_PREDICTED_RESULTS"}
Thought:The prediction has been successfully completed. The results are stored in the table 'my_automl_model_1_PREDICTED_RESULTS'. Now, I will provide this information to the user.
Action:
```
{
  "action": "Final Answer",
  "action_input": "The prediction has been successfully completed. The results ar

"The prediction has been successfully completed. The results are stored in the table 'my_automl_model_1_PREDICTED_RESULTS'."

In [13]:
chatbot.run("show me the predicted result")



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "fetch_data",
  "action_input": {
    "table_name": "my_automl_model_1_PREDICTED_RESULTS",
    "last_n": 10
  }
}
```
Observation:    ID              SCORES
0  31   384.1198852086927
1  32  460.26114238073586
2  33    402.603574262169
3  34   478.7448314323638
4  35   421.0872633119485
5  36    497.228520480295


> Finished chain.


,ID,SCORES
0,31,384.1198852086927
1,32,460.26114238073586
2,33,402.603574262169
3,34,478.7448314323638
4,35,421.0872633119485
5,36,497.228520480295


##### Plot the predictions using the tool `forecast_line_plot`

In [14]:
chatbot.run("Generate the line plot on the predictions and compared with the actual table SHAMPOO_SALES_ACTUAL")



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "forecast_line_plot",
  "action_input": {
    "predict_table_name": "my_automl_model_1_PREDICTED_RESULTS",
    "actual_table_name": "SHAMPOO_SALES_ACTUAL"
  }
}
```


Observation: {"html_file": "C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_chart\\my_automl_model_1_PREDICTED_RESULTS_forecast_line_plot.html"}
Thought:The line plot has been successfully generated and saved as an HTML file. The file path is "C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_chart\\my_automl_model_1_PREDICTED_RESULTS_forecast_line_plot.html". This plot visualizes the forecasted results from the model 'my_automl_model_1' and compares them with the actual data from the table 'SHAMPOO_SALES_ACTUAL'. 
Action:
```
{
  "action": "Final Answer",
  "action_input": "The line plot has been successfully generated and saved as an HTML file. The file path is \"C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_chart\\my_automl_model_1_PREDICTED_RESULTS_forecast_line_plot.html\". This plot visualizes the forecasted results from the model 'my_automl_model_1' and compares them with the actual data from the table 'SHAMPOO_SALES_ACTUAL'."
}
```

> Finished chain.


'The line plot has been successfully generated and saved as an HTML file. The file path is "C:\\Users\\I308290\\AppData\\Local\\Temp\\hanaml_chart\\my_automl_model_1_PREDICTED_RESULTS_forecast_line_plot.html". This plot visualizes the forecasted results from the model \'my_automl_model_1\' and compares them with the actual data from the table \'SHAMPOO_SALES_ACTUAL\'.'

##### Generate CAP artifacts using the tool `cap_artifacts`

In [15]:
chatbot.run("I want to generate CAP artifacts, the project name is my_project and output path is /tmp")



> Entering new AgentExecutor chain...
Thought: The user wants to generate CAP artifacts for a project named "my_project" and save them to the "/tmp" directory. However, the user didn't specify the model name and version. I need to ask for these details before proceeding. 
Action:
```
{
  "action": "Final Answer",
  "action_input": "Could you please provide the name and version of the model for which you want to generate CAP artifacts?"
}
```

> Finished chain.


'Could you please provide the name and version of the model for which you want to generate CAP artifacts?'

In [19]:
chatbot.run("'my_automl_model' with version 1")



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "cap_artifacts",
  "action_input": {
    "name": "my_automl_model",
    "version": 1,
    "project_name": "my_project",
    "output_dir": "/tmp"
  }
}
```
Observation: CAP artifacts generated successfully. Root directory: /tmp\my_project
Thought:The CAP artifacts for the model 'my_automl_model' with version 1 have been successfully generated under the project name 'my_project'. The root directory for the generated artifacts is '/tmp\my_project'.

> Finished chain.


"The CAP artifacts for the model 'my_automl_model' with version 1 have been successfully generated under the project name 'my_project'. The root directory for the generated artifacts is '/tmp\\my_project'."

In [20]:
chatbot.memory.messages

[HumanMessage(content='Show me the last 10 records of data from SHAMPOO_SALES_DATA_TBL'),
 AIMessage(content='Action:\n{\n  "action": "fetch_data",\n  "action_input": {\n    "table_name": "SHAMPOO_SALES_DATA_TBL",\n    "last_n": 10\n  }\n}'),
 AIMessage(content='The returned is a pandas dataframe with the metadata:\n{"columns": ["ID", "SALES"], "dtypes": ["int64", "float64"]}'),
 HumanMessage(content='Then maybe create a dataset report for me on this table'),
 AIMessage(content='Action:\n{\n  "action": "ts_dataset_report",\n  "action_input": {\n    "table_name": "SHAMPOO_SALES_DATA_TBL",\n    "key": "ID",\n    "endog": "SALES"\n  }\n}'),
 AIMessage(content='The question is `Then maybe create a dataset report for me on this table`. The tool ts_dataset_report has been already called via {\'table_name\': \'SHAMPOO_SALES_DATA_TBL\', \'key\': \'ID\', \'endog\': \'SALES\'}. The result is `{"html_file": "C:\\\\Users\\\\I308290\\\\AppData\\\\Local\\\\Temp\\\\hanaml_report\\\\SHAMPOO_SALES_DATA